In [ ]:
# Uncomment if needed
# !pip install torch torchvision Pillow numpy matplotlib

In [ ]:
# Imports
import sys
import os
import math

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import torchvision.transforms as transforms

In [ ]:
from flow import Glow

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
plt.rcParams.update({
    "figure.figsize": (6, 4),
    "figure.dpi": 120,

    "font.family": "serif",
    "font.serif": ["Times New Roman"],

    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,

    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    "axes.linewidth": 1.0,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",

    "legend.fontsize": 10,
    "legend.frameon": False,

    "lines.linewidth": 2,

    "savefig.bbox": "tight",
})

In [ ]:
latent_data_file = "./latents.npz"
flow_model_file = "./flow_best.pth"

In [ ]:
# load data
data = np.load(latent_data_file)
latents_tensor = torch.from_numpy(data['latents'].astype(np.float32))
latents_mean_tensor = torch.from_numpy(data['latents_mean'].astype(np.float32))
latents_std_tensor = torch.from_numpy(data['latents_std'].astype(np.float32))

latents_tensor = (latents_tensor - latents_mean_tensor) / latents_std_tensor

latent_dataset = TensorDataset(latents_tensor)
train_size = int(0.9 * len(latent_dataset))
test_size  = len(latent_dataset) - train_size

train_split, test_split = random_split(latent_dataset, [train_size, test_size])

latent_train_loader = DataLoader(train_split, batch_size=64, shuffle=True,  num_workers=0)
latent_test_loader  = DataLoader(test_split,  batch_size=64, shuffle=False, num_workers=0)

In [ ]:
# load flow
flow = torch.load(flow_model_file, weights_only=False)
flow.to(device)
flow.eval()

In [ ]:
# flow statistics (mean and std)
def base_stats(flow, dataloader, device):
    flow.eval()
    zs = []

    with torch.no_grad():
        for (x,) in dataloader:
            x = x.to(device)

            z_list, _ = flow(x)

            # concatenate all scale latents
            z_flat = torch.cat(
                [z.reshape(z.size(0), -1) for z in z_list],
                dim=1
            )

            zs.append(z_flat.cpu())

    z = torch.cat(zs, dim=0)

    print("Base latent statistics:")
    print("mean:", z.mean().item())
    print("std:", z.std().item())

In [ ]:
base_stats(flow, latent_train_loader, device)

In [ ]:
# Latent vs output
def plot_latent_vs_output(flow, dataloader, device):
    flow.eval()
    zs = []
    x_outs = []

    with torch.no_grad():
        for (x,) in dataloader:
            x = x.to(device)

            # Forward pass: get latents
            z_list, _ = flow(x)
            z_flat = torch.cat([z.reshape(z.size(0), -1) for z in z_list], dim=1)
            zs.append(z_flat.cpu())

            # Optional: map latent back to data space
            x_recon = flow.inverse(z_list)  # depends on your flow implementation
            x_outs.append(x_recon.cpu().reshape(x_recon.size(0), -1))

    # Combine batches
    zs = torch.cat(zs, dim=0).reshape(-1).numpy()
    x_outs = torch.cat(x_outs, dim=0).reshape(-1).numpy()

    # Plot base-space latent
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.hist(zs, bins=100, density=True, alpha=0.6)
    x_vals = np.linspace(-5, 5, 500)
    plt.plot(x_vals, norm.pdf(x_vals, 0, 1))
    plt.title("Base latent distribution")
    plt.xlabel("z")
    plt.ylabel("density")

    # Plot output-space distribution
    plt.subplot(1,2,2)
    plt.hist(x_outs, bins=100, density=True, alpha=0.6)
    plt.title("Output-space distribution")
    plt.xlabel("x_hat")
    plt.ylabel("density")

    plt.show()

In [ ]:
plot_latent_vs_output(flow, latent_train_loader, device)

In [ ]:
# latent vs reconstructed vs original data
def plot_flow_distributions(flow, dataloader, device):
    """
    Plots three distributions:
    1. Base latent z (should be ~N(0,1))
    2. Reconstructed output x_hat
    3. Original data x
    """
    flow.eval()
    
    zs = []
    x_hats = []
    x_all = []

    with torch.no_grad():
        for (x,) in dataloader:
            x = x.to(device)
            x_all.append(x.cpu().reshape(x.size(0), -1))

            # Forward pass: get latent(s)
            z_list, _ = flow(x)
            z_flat = torch.cat([z.reshape(z.size(0), -1) for z in z_list], dim=1)
            zs.append(z_flat.cpu())

            # Inverse pass: reconstruct x_hat
            # Some flows accept a list of latents; adjust if your flow expects a single tensor
            x_hat = flow.inverse(z_list)
            x_hats.append(x_hat.cpu().reshape(x_hat.size(0), -1))

    # Combine batches
    zs = torch.cat(zs, dim=0).reshape(-1).numpy()
    x_hats = torch.cat(x_hats, dim=0).reshape(-1).numpy()
    x_all = torch.cat(x_all, dim=0).reshape(-1).numpy()

    # Plot
    plt.figure(figsize=(18,5))

    # Base latent z
    plt.subplot(1,3,1)
    plt.hist(zs, bins=100, density=True, alpha=0.6, color='skyblue', label='latent z')
    x_vals = np.linspace(-5, 5, 500)
    plt.plot(x_vals, norm.pdf(x_vals, 0, 1), color='red', lw=2, label='N(0,1)')
    plt.title("Base latent distribution")
    plt.xlabel("z value")
    plt.ylabel("density")
    plt.legend()

    # Reconstructed x_hat
    plt.subplot(1,3,2)
    plt.hist(x_hats, bins=100, density=True, alpha=0.6, color='lightgreen', label='x_hat')
    plt.title("Reconstructed output distribution")
    plt.xlabel("x_hat value")
    plt.ylabel("density")
    plt.legend()

    # Original x
    plt.subplot(1,3,3)
    plt.hist(x_all, bins=100, density=True, alpha=0.6, color='lightcoral', label='x')
    plt.title("Original data distribution")
    plt.xlabel("x value")
    plt.ylabel("density")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_flow_distributions(flow, latent_train_loader, device)

In [ ]:
def plot_full_latent_diagnostics(flow, dataloader, device, tsne_perplexity=30):
    """
    Full latent-space diagnostic for normalizing flows.
    
    Plots:
    1. First two latent dimensions (2D scatter)
    2. t-SNE projection of full latent vector
    
    Args:
        flow: trained normalizing flow model
        dataloader: PyTorch DataLoader, provides (x, y) if labels_available=True
        device: torch.device
        labels_available: bool, whether dataset provides class labels
        tsne_perplexity: int, t-SNE perplexity parameter
    """
    flow.eval()
    zs = []
    ys = []

    with torch.no_grad():
        for batch in dataloader:
            x = batch[0]
            x = x.to(device)

            # Forward pass
            z_list, _ = flow(x)
            z_flat = torch.cat([z.reshape(z.size(0), -1) for z in z_list], dim=1)
            zs.append(z_flat.cpu())

    zs = torch.cat(zs, dim=0).numpy()

    # Compute t-SNE
    tsne = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=42)
    zs_tsne = tsne.fit_transform(zs)

    # Plot
    plt.figure(figsize=(14,6))

    # First two latent dims
    plt.subplot(1,2,1)
    plt.scatter(zs[:,0], zs[:,1], color='blue', s=10, alpha=0.7)
    plt.xlabel("z dim 1")
    plt.ylabel("z dim 2")
    plt.title("Latent space: first two dimensions")
    plt.grid(True)

    # t-SNE of full latent
    plt.subplot(1,2,2)
    plt.scatter(zs_tsne[:,0], zs_tsne[:,1], color='green', s=10, alpha=0.7)
    plt.xlabel("t-SNE dim 1")
    plt.ylabel("t-SNE dim 2")
    plt.title("Latent space: t-SNE projection")
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_full_latent_diagnostics(flow, latent_train_loader, device)